# **Projeção e Modelagem Preditiva da Coleta de Resíduos Sólidos Urbanos em São Paulo de 2013–2027**

# 1. SETUP E IMPORTAÇÃO DE BIBLIOTECAS

In [13]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from prophet import Prophet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
)
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings('ignore')

# 2. CARREGAMENTO, TRATAMENTO E ENGENHARIA DE FEATURES

In [14]:
caminho_arquivo = 'DF/total_mensal_fixed_ESTIMATIVA.xlsx'

# Leitura e transposição
df_raw = pd.read_excel(caminho_arquivo, header=None)
df_raw.columns = df_raw.iloc[0]
df_raw = df_raw.drop(index=0)

df_transposto = df_raw.transpose().reset_index()
df_transposto.columns = ['mes_ano', 'y']

# Mapeamento de meses em português
meses_map = {
    'jan': '01',
    'fev': '02',
    'mar': '03',
    'abr': '04',
    'mai': '05',
    'jun': '06',
    'jul': '07',
    'ago': '08',
    'set': '09',
    'out': '10',
    'nov': '11',
    'dez': '12',
}


def converter_data_pt(data_str):
  try:
    mes, ano = str(data_str).strip().split('/')
    return f'20{ano}-{meses_map[mes.lower()]}-01'
  except:
    return np.nan


def limpar_valor(val):
  if pd.isna(val) or isinstance(val, (int, float)):
    return val
  s = str(val).strip()
  if ',' in s:
    s = s.replace('.', '').replace(',', '.')
  return float(s)


# Tratamento de datas e valores numéricos
df_transposto['ds'] = pd.to_datetime(
    df_transposto['mes_ano'].apply(converter_data_pt), errors='coerce'
)
df_transposto['y'] = df_transposto['y'].apply(limpar_valor)

# Base limpa sequencial (Prophet)
df_clean = (
    df_transposto.dropna(subset=['ds', 'y'])
    .sort_values(by='ds')
    .reset_index(drop=True)[['ds', 'y']]
)

# Criação de 24 Lags Temporais para Random Forest
df_model = df_clean.copy()
for lag in range(1, 25):
  df_model[f'lag_{lag}'] = df_model['y'].shift(lag)

df_model = df_model.dropna().reset_index(drop=True)

print(f'Base tratada com sucesso! Período: {df_clean["ds"].min().strftime("%Y-%m")} até {df_clean["ds"].max().strftime("%Y-%m")}')

Base tratada com sucesso! Período: 2013-01 até 2020-12


# 3. EXPERIMENTO 1: VALIDAÇÃO OUT-OF-SAMPLE (2019 - 2020)

In [15]:
# Slice de Teste Comum para Todos os Modelos
df_test_val = df_clean[
    (df_clean['ds'] >= '2019-01-01') & (df_clean['ds'] <= '2020-12-01')
].copy()
y_test_real = df_test_val['y'].values

# ------------------------------------------
# A) RANDOM FOREST (Treino Restrito: 2015-2018)
# ------------------------------------------
df_train_rf_val = df_model[
    (df_model['ds'] >= '2015-01-01') & (df_model['ds'] <= '2018-12-01')
].copy()
X_train_rf_val = df_train_rf_val[[f'lag_{i}' for i in range(1, 25)]]
y_train_rf_val = df_train_rf_val['y']

rf_val = RandomForestRegressor(n_estimators=100, random_state=42)
rf_val.fit(X_train_rf_val, y_train_rf_val)

last_known_val = y_train_rf_val.values[-24:].copy()
future_preds_rf_val = []

for _ in range(24):
  X_future = last_known_val[-24:].reshape(1, -1)
  next_pred = rf_val.predict(X_future)[0]
  future_preds_rf_val.append(next_pred)
  last_known_val = np.append(last_known_val, next_pred)

mae_rf_val = mean_absolute_error(y_test_real, future_preds_rf_val)
rmse_rf_val = np.sqrt(mean_squared_error(y_test_real, future_preds_rf_val))
mape_rf_val = (
    mean_absolute_percentage_error(y_test_real, future_preds_rf_val) * 100
)

# ------------------------------------------
# B) ARIMA (24, 0, 3) (Treino Restrito: 2015-2018)
# ------------------------------------------
model_arima_val = ARIMA(y_train_rf_val, order=(24, 0, 3))
fit_arima_val = model_arima_val.fit()
preds_arima_val = fit_arima_val.forecast(steps=24)

mae_arima_val = mean_absolute_error(y_test_real, preds_arima_val)
rmse_arima_val = np.sqrt(mean_squared_error(y_test_real, preds_arima_val))
mape_arima_val = (
    mean_absolute_percentage_error(y_test_real, preds_arima_val) * 100
)

# ------------------------------------------
# C) PROPHET (Treino Restrito: 2015-2018)
# ------------------------------------------
df_train_prophet_15_18 = df_clean[
    (df_clean['ds'] >= '2015-01-01') & (df_clean['ds'] <= '2018-12-01')
].copy()

model_prophet_15_18 = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    interval_width=0.95,
)
model_prophet_15_18.fit(df_train_prophet_15_18)

future_15_18 = model_prophet_15_18.make_future_dataframe(
    periods=24, freq='MS'
)
forecast_15_18 = model_prophet_15_18.predict(future_15_18)
preds_prophet_15_18 = forecast_15_18[forecast_15_18['ds'] >= '2019-01-01'][
    'yhat'
].values

mae_15_18 = mean_absolute_error(y_test_real, preds_prophet_15_18)
rmse_15_18 = np.sqrt(mean_squared_error(y_test_real, preds_prophet_15_18))
mape_15_18 = mean_absolute_percentage_error(y_test_real, preds_prophet_15_18) * 100

print('Validação Out-of-Sample concluída para todos os modelos! \n')

print("-----------------------------------------------------------------------------------------")
print(" 1. TESTE OUT-OF-SAMPLE (2019 - 2020)")
print(" (Modelos Treinados com Dados Passados vs Dados Reais de Validação 2019-2020)")
print("-----------------------------------------------------------------------------------------")

print("\n --- RANDOM FOREST (Treino: 2015 - 2018) ---")
print(f" MAE  (Erro Absoluto Médio):       {mae_rf_val:,.2f} toneladas")
print(f" RMSE (Raiz do Erro Quadrático):   {rmse_rf_val:,.2f} toneladas")
print(f" MAPE (Erro Percentual Médio):     {mape_rf_val:.2f}%")

print("\n --- ARIMA (24, 0, 3) (Treino: 2015 - 2018) ---")
print(f" MAE  (Erro Absoluto Médio):       {mae_arima_val:,.2f} toneladas")
print(f" RMSE (Raiz do Erro Quadrático):   {rmse_arima_val:,.2f} toneladas")
print(f" MAPE (Erro Percentual Médio):     {mape_arima_val:.2f}%")

print("\n --- PROPHET (Treino Restrito: 2015 - 2018) ---")
print(f" MAE  (Erro Absoluto Médio):       {mae_15_18:,.2f} toneladas")
print(f" RMSE (Raiz do Erro Quadrático):   {rmse_15_18:,.2f} toneladas")
print(f" MAPE (Erro Percentual Médio):     {mape_15_18:.2f}%")

09:56:24 - cmdstanpy - INFO - Chain [1] start processing
09:56:24 - cmdstanpy - INFO - Chain [1] done processing


Validação Out-of-Sample concluída para todos os modelos! 

-----------------------------------------------------------------------------------------
 1. TESTE OUT-OF-SAMPLE (2019 - 2020)
 (Modelos Treinados com Dados Passados vs Dados Reais de Validação 2019-2020)
-----------------------------------------------------------------------------------------

 --- RANDOM FOREST (Treino: 2015 - 2018) ---
 MAE  (Erro Absoluto Médio):       19,292.87 toneladas
 RMSE (Raiz do Erro Quadrático):   23,330.07 toneladas
 MAPE (Erro Percentual Médio):     4.08%

 --- ARIMA (24, 0, 3) (Treino: 2015 - 2018) ---
 MAE  (Erro Absoluto Médio):       30,590.86 toneladas
 RMSE (Raiz do Erro Quadrático):   35,669.99 toneladas
 MAPE (Erro Percentual Médio):     6.46%

 --- PROPHET (Treino Restrito: 2015 - 2018) ---
 MAE  (Erro Absoluto Médio):       18,155.30 toneladas
 RMSE (Raiz do Erro Quadrático):   21,630.88 toneladas
 MAPE (Erro Percentual Médio):     3.91%


# 4. VISUALIZAÇÃO OUT-OF-SAMPLE: MODELOS VS REAL (2019-2020)

In [16]:
fig_val = go.Figure()

# 1. DADOS REAIS DE VALIDAÇÃO (2019-2020)
fig_val.add_trace(
    go.Scatter(
        x=df_test_val['ds'],
        y=y_test_real,
        mode='lines+markers',
        name='Dados Reais (2019-2020)',
        line=dict(color='black', width=2.5),
        marker=dict(size=6, symbol='circle'),
    )
)

# 2. RANDOM FOREST
fig_val.add_trace(
    go.Scatter(
        x=df_test_val['ds'],
        y=future_preds_rf_val,
        mode='lines+markers',
        name=f'Random Forest (MAPE: {mape_rf_val:.2f}%)',
        line=dict(color='#5CB85C', dash='dash', width=2),
        marker=dict(size=5, symbol='square'),
    )
)

# 3. ARIMA (24, 0, 3)
fig_val.add_trace(
    go.Scatter(
        x=df_test_val['ds'],
        y=preds_arima_val,
        mode='lines+markers',
        name=f'ARIMA (24,0,3) (MAPE: {mape_arima_val:.2f}%)',
        line=dict(color='#D9534F', dash='dash', width=2),
        marker=dict(size=5, symbol='diamond'),
    )
)

# 4. PROPHET
fig_val.add_trace(
    go.Scatter(
        x=df_test_val['ds'],
        y=preds_prophet_15_18,
        mode='lines+markers',
        name=f'Prophet (MAPE: {mape_15_18:.2f}%)',
        line=dict(color='#0275D8', dash='dash', width=2),
        marker=dict(size=5, symbol='triangle-up'),
    )
)

# Layout e Design do Zoom de Validação
fig_val.update_layout(
    title=dict(
        text=(
            '<b>Validação Out-of-Sample (2019–2020)</b><br><sup>Comparativo'
            ' de Desempenho: Modelos Treinados (2015-2018) vs Dados Reais'
            ' Observados</sup>'
        ),
        font=dict(size=18),
    ),
    xaxis_title='Mês/Ano',
    yaxis_title='Toneladas Coletadas',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='Gray',
        borderwidth=1,
    ),
    xaxis=dict(
        dtick='M1',
        tickformat='%b/%Y',
        showgrid=True,
        gridcolor='LightGray',
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='LightGray',
        tickformat=',.0f',
    ),
    margin=dict(l=50, r=150, t=90, b=50),
)

fig_val.show()

# 5. EXPERIMENTO 2 & 3: AJUSTE IN-SAMPLE E PROJEÇÃO FINAL (2021-2027)

In [17]:
steps = 84  # 7 anos (2021 a 2027)
future_dates = pd.date_range(start='2021-01-01', periods=steps, freq='MS')

# Bases completas até 2020
df_model_train_full = df_model[df_model['ds'] <= '2020-12-01'].copy()
df_clean_train_full = df_clean[df_clean['ds'] <= '2020-12-01'].copy()

# ------------------------------------------
# A) RANDOM FOREST (Final + In-Sample Fit)
# ------------------------------------------
X_full = df_model_train_full[[f'lag_{i}' for i in range(1, 25)]]
y_full_rf = df_model_train_full['y']

rf_model_final = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_final.fit(X_full, y_full_rf)

# Ajuste In-Sample Random Forest
y_fitted_rf_full = rf_model_final.predict(X_full)
mae_rf_full = mean_absolute_error(y_full_rf, y_fitted_rf_full)
rmse_rf_full = np.sqrt(mean_squared_error(y_full_rf, y_fitted_rf_full))
mape_rf_full = (
    mean_absolute_percentage_error(y_full_rf, y_fitted_rf_full) * 100
)

# Projeção Random Forest 2021-2027
last_known_full = y_full_rf.values[-24:].copy()
future_preds_rf_final = []

for _ in range(steps):
    X_future = last_known_full[-24:].reshape(1, -1)
    next_pred = rf_model_final.predict(X_future)[0]
    future_preds_rf_final.append(next_pred)
    last_known_full = np.append(last_known_full, next_pred)

# ------------------------------------------
# B) ARIMA (24,0,3) (Final - Treino Completo 2013-2020)
# ------------------------------------------
y_full_arima = df_clean_train_full['y']

model_arima_final = ARIMA(y_full_arima, order=(24, 0, 3))
fit_arima_final = model_arima_final.fit()

# Ajuste In-Sample ARIMA
y_fitted_arima_full = fit_arima_final.fittedvalues
mae_arima_full = mean_absolute_error(y_full_arima, y_fitted_arima_full)
rmse_arima_full = np.sqrt(
    mean_squared_error(y_full_arima, y_fitted_arima_full)
)
mape_arima_full = (
    mean_absolute_percentage_error(y_full_arima, y_fitted_arima_full) * 100
)

# Projeção ARIMA 2021-2027
previsao_arima_final = fit_arima_final.forecast(steps=steps).values

# ------------------------------------------
# C) PROPHET (Final + In-Sample Fit)
# ------------------------------------------
model_prophet_final = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.01,
    interval_width=0.95,
)
model_prophet_final.fit(df_clean_train_full)

# Ajuste In-Sample Prophet
in_sample_pred_p = model_prophet_final.predict(df_clean_train_full)[
    'yhat'
].values
mae_in_sample_p = mean_absolute_error(
    df_clean_train_full['y'], in_sample_pred_p
)
rmse_in_sample_p = np.sqrt(
    mean_squared_error(df_clean_train_full['y'], in_sample_pred_p)
)
mape_in_sample_p = (
    mean_absolute_percentage_error(df_clean_train_full['y'], in_sample_pred_p)
    * 100
)

# Projeção Prophet 2021-2027
future_final = model_prophet_final.make_future_dataframe(
    periods=steps, freq='MS'
)
forecast_final = model_prophet_final.predict(future_final)

df_previsoes_prophet_final = forecast_final[
    forecast_final['ds'] >= '2021-01-01'
][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()

# ------------------------------------------
# D) CONSISTÊNCIA E UNIFICAÇÃO DAS PREVISÕES
# ------------------------------------------
df_previsoes_consolidado = pd.DataFrame({
    'ds': future_dates,
    'Previsao_ARIMA': previsao_arima_final,
    'Previsao_RandomForest': future_preds_rf_final,
    'Previsao_Prophet': df_previsoes_prophet_final['yhat'].values,
    'Prophet_Limite_Inferior_95': df_previsoes_prophet_final[
        'yhat_lower'
    ].values,
    'Prophet_Limite_Superior_95': df_previsoes_prophet_final[
        'yhat_upper'
    ].values,
})

print('Retreinamento completo e projeções 2021-2027 geradas com sucesso!')

09:56:27 - cmdstanpy - INFO - Chain [1] start processing
09:56:27 - cmdstanpy - INFO - Chain [1] done processing


Retreinamento completo e projeções 2021-2027 geradas com sucesso!


# 6. RELATÓRIO CONSOLIDADO DE DESEMPENHO
* **Validação Out-of-Sample (2019–2020):** O modelo Prophet apresentou a maior capacidade de generalização com o menor erro nos testes

* **Ajuste In-Sample e o Risco das Métricas:** O Random Forest registrou as métricas aparentes mais baixas no ajuste *In-Sample*, como o MAPE = 1,47%. Contudo, assim como visível no bloco 7, isso ocorre pois o modelo converge rapidamente para a média histórica em torno de ~470.000 toneladas, e não uma capacidade preditiva superior.

In [18]:
print("=========================================================================================")
print("RELATÓRIO CONSOLIDADO DE DESEMPENHO DOS MODELOS")
print("=========================================================================================\n")

print("-----------------------------------------------------------------------------------------")
print(" 1. TESTE OUT-OF-SAMPLE (2019 - 2020)")
print(" (Modelos Treinados com Dados Passados vs Dados Reais de Validação 2019-2020)")
print("-----------------------------------------------------------------------------------------")

print("\n --- RANDOM FOREST (Treino: 2015 - 2018) ---")
print(f" MAE  (Erro Absoluto Médio):       {mae_rf_val:,.2f} toneladas")
print(f" RMSE (Raiz do Erro Quadrático):   {rmse_rf_val:,.2f} toneladas")
print(f" MAPE (Erro Percentual Médio):     {mape_rf_val:.2f}%")

print("\n --- ARIMA (24, 0, 3) (Treino: 2015 - 2018) ---")
print(f" MAE  (Erro Absoluto Médio):       {mae_arima_val:,.2f} toneladas")
print(f" RMSE (Raiz do Erro Quadrático):   {rmse_arima_val:,.2f} toneladas")
print(f" MAPE (Erro Percentual Médio):     {mape_arima_val:.2f}%")

print("\n --- PROPHET (Treino Restrito: 2015 - 2018) ---")
print(f" MAE  (Erro Absoluto Médio):       {mae_15_18:,.2f} toneladas")
print(f" RMSE (Raiz do Erro Quadrático):   {rmse_15_18:,.2f} toneladas")
print(f" MAPE (Erro Percentual Médio):     {mape_15_18:.2f}%")

print("\n-----------------------------------------------------------------------------------------")
print(" 2. AJUSTE IN-SAMPLE (RECONSTRUÇÃO DA BASE HISTÓRICA COMPLETA)")
print(" (Desempenho de Ajuste do Modelo Sobre os Próprios Dados de Treino)")
print("-----------------------------------------------------------------------------------------")

print('\n --- RANDOM FOREST (Base Completa) ---')
print(f' MAE  (Erro Absoluto Médio):       {mae_rf_full:,.2f} toneladas')
print(f' RMSE (Raiz do Erro Quadrático):   {rmse_rf_full:,.2f} toneladas')
print(f' MAPE (Erro Percentual Médio):     {mape_rf_full:.2f}%')

print("\n --- ARIMA (24, 0, 3) (Base Completa) ---")
print(f" MAE  (Erro Absoluto Médio):       {mae_arima_full:,.2f} toneladas")
print(f" RMSE (Raiz do Erro Quadrático):   {rmse_arima_full:,.2f} toneladas")
print(f" MAPE (Erro Percentual Médio):     {mape_arima_full:.2f}%")

print("\n --- PROPHET (Base Completa) ---")
print(f" MAE  (Erro Absoluto Médio):       {mae_in_sample_p:,.2f} toneladas")
print(f" RMSE (Raiz do Erro Quadrático):   {rmse_in_sample_p:,.2f} toneladas")
print(f" MAPE (Erro Percentual Médio):     {mape_in_sample_p:.2f}%")

print("\n=========================================================================================")

RELATÓRIO CONSOLIDADO DE DESEMPENHO DOS MODELOS

-----------------------------------------------------------------------------------------
 1. TESTE OUT-OF-SAMPLE (2019 - 2020)
 (Modelos Treinados com Dados Passados vs Dados Reais de Validação 2019-2020)
-----------------------------------------------------------------------------------------

 --- RANDOM FOREST (Treino: 2015 - 2018) ---
 MAE  (Erro Absoluto Médio):       19,292.87 toneladas
 RMSE (Raiz do Erro Quadrático):   23,330.07 toneladas
 MAPE (Erro Percentual Médio):     4.08%

 --- ARIMA (24, 0, 3) (Treino: 2015 - 2018) ---
 MAE  (Erro Absoluto Médio):       30,590.86 toneladas
 RMSE (Raiz do Erro Quadrático):   35,669.99 toneladas
 MAPE (Erro Percentual Médio):     6.46%

 --- PROPHET (Treino Restrito: 2015 - 2018) ---
 MAE  (Erro Absoluto Médio):       18,155.30 toneladas
 RMSE (Raiz do Erro Quadrático):   21,630.88 toneladas
 MAPE (Erro Percentual Médio):     3.91%

---------------------------------------------------------

# 7. VISUALIZAÇÃO GRÁFICA COMPARATIVA UNIFICADA

In [19]:
df_historico_plot = df_clean[df_clean['ds'] <= '2020-12-01'].sort_values(
    by='ds'
)

# Máscaras Temporais para as Fases
mask_est = (df_previsoes_consolidado['ds'] >= '2021-01-01') & (
    df_previsoes_consolidado['ds'] <= '2025-12-01'
)
mask_fut = df_previsoes_consolidado['ds'] >= '2026-01-01'

fig = go.Figure()

# 1. DADOS HISTÓRICOS REAIS (2013 - 2020)
fig.add_trace(
    go.Scatter(
        x=df_historico_plot['ds'],
        y=df_historico_plot['y'],
        mode='lines',
        name='Histórico Real (2013-2020)',
        legendgroup='historico',
        line=dict(color='black', width=2),
    )
)

# 2. ESTIMATIVA INTERMEDIÁRIA (2021 - 2025)
fig.add_trace(
    go.Scatter(
        x=df_previsoes_consolidado.loc[mask_est, 'ds'],
        y=df_previsoes_consolidado.loc[mask_est, 'Previsao_ARIMA'],
        mode='lines',
        name='Modelo ARIMA',
        legendgroup='arima',
        showlegend=True,
        line=dict(color='#D9534F', dash='dot', width=2),
    )
)

fig.add_trace(
    go.Scatter(
        x=df_previsoes_consolidado.loc[mask_est, 'ds'],
        y=df_previsoes_consolidado.loc[mask_est, 'Previsao_RandomForest'],
        mode='lines',
        name='Modelo Random Forest',
        legendgroup='rf',
        showlegend=True,
        line=dict(color='#5CB85C', dash='dot', width=2),
    )
)

fig.add_trace(
    go.Scatter(
        x=df_previsoes_consolidado.loc[mask_est, 'ds'],
        y=df_previsoes_consolidado.loc[mask_est, 'Previsao_Prophet'],
        mode='lines',
        name='Modelo Prophet',
        legendgroup='prophet',
        showlegend=True,
        line=dict(color='#0275D8', dash='dot', width=2),
    )
)

# 3. PROJEÇÃO FUTURA (2026 - 2027)
fig.add_trace(
    go.Scatter(
        x=df_previsoes_consolidado.loc[mask_fut, 'ds'],
        y=df_previsoes_consolidado.loc[mask_fut, 'Previsao_ARIMA'],
        mode='lines',
        name='Modelo ARIMA',
        legendgroup='arima',
        showlegend=False,
        line=dict(color='#D9534F', dash='dot', width=2.5),
    )
)

fig.add_trace(
    go.Scatter(
        x=df_previsoes_consolidado.loc[mask_fut, 'ds'],
        y=df_previsoes_consolidado.loc[mask_fut, 'Previsao_RandomForest'],
        mode='lines',
        name='Modelo Random Forest',
        legendgroup='rf',
        showlegend=False,
        line=dict(color='#5CB85C', dash='dot', width=2.5),
    )
)

fig.add_trace(
    go.Scatter(
        x=df_previsoes_consolidado.loc[mask_fut, 'ds'],
        y=df_previsoes_consolidado.loc[mask_fut, 'Previsao_Prophet'],
        mode='lines',
        name='Modelo Prophet',
        legendgroup='prophet',
        showlegend=False,
        line=dict(color='#0275D8', dash='dot', width=2.5),
    )
)

# 4. ELEMENTOS DE DESIGN E MARCADORES TEMPORAIS
fig.add_vline(
    x='2021-01-01', line_dash='dot', line_color='gray', line_width=1.5
)
fig.add_vline(
    x='2026-01-01', line_dash='dot', line_color='gray', line_width=1.5
)

fig.add_annotation(
    x='2021-01-01',
    y=1.02,
    yref='paper',
    text='Início Estimativa (2021)',
    showarrow=False,
    xanchor='right',
    yanchor='bottom',
    font=dict(color='gray', size=11),
)

fig.add_annotation(
    x='2026-01-01',
    y=1.02,
    yref='paper',
    text='Início Projeção (2026)',
    showarrow=False,
    xanchor='right',
    yanchor='bottom',
    font=dict(color='gray', size=11),
)

fig.update_layout(
    title=dict(
        text=(
            '<b>Comparativo de Modelos de Coleta RSU em SP</b><br><sup>Histórico'
            ' Real, Estimativa Intermediária (2021-2025) e Projeção'
            ' (2026-2027)</sup>'
        ),
        font=dict(size=18),
    ),
    xaxis_title='Ano/Mês',
    yaxis_title='Toneladas Coletadas',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='Gray',
        borderwidth=1,
    ),
    xaxis=dict(
        dtick='M12',
        tickformat='%Y',
        showgrid=True,
        gridcolor='LightGray',
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='LightGray',
        tickformat=',.0f',
    ),
    margin=dict(l=50, r=150, t=90, b=50),
)

fig.show()

# 8. VISUALIZAÇÃO TÉCNICA DA PROJEÇÃO PROPHET

* Exibição da trajetória do modelo Prophet acompanhada do intervalo de confiança de 95%, delimitando a margem de incerteza de máximo e mínimo, projetados até 2027.

In [20]:
# Isolando dados históricos (2013 - 2020)
df_hist_prophet = df_clean[df_clean['ds'] <= '2020-12-01'].sort_values(
    by='ds'
)

# Máscaras de Fase
est_mask = (df_previsoes_consolidado['ds'] >= '2021-01-01') & (
    df_previsoes_consolidado['ds'] <= '2025-12-01'
)
fut_mask = df_previsoes_consolidado['ds'] >= '2026-01-01'

# Slices de dados
df_est = df_previsoes_consolidado.loc[est_mask]
df_fut = df_previsoes_consolidado.loc[fut_mask]

fig_prophet = go.Figure()

# -------------------------------------------------------------------------
# FASE 1: DADOS HISTÓRICOS REAIS (2013 - 2020) -> Linha Preta
# -------------------------------------------------------------------------
fig_prophet.add_trace(
    go.Scatter(
        x=df_hist_prophet['ds'],
        y=df_hist_prophet['y'],
        mode='lines',
        name='1. Histórico Real (2013-2020)',
        line=dict(color='black', width=2),
    )
)

# -------------------------------------------------------------------------
# FASE 2: ESTIMATIVA PROPHET (2021 - 2025) -> Linha Amarela + Faixa 95%
# -------------------------------------------------------------------------
# Faixa de Incerteza (Amarelo Translúcido)
fig_prophet.add_trace(
    go.Scatter(
        x=pd.concat([df_est['ds'], df_est['ds'][::-1]]),
        y=pd.concat([
            df_est['Prophet_Limite_Superior_95'],
            df_est['Prophet_Limite_Inferior_95'][::-1],
        ]),
        fill='toself',
        fillcolor='rgba(255, 165, 0, 0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo='skip',
        showlegend=False,
        name='Intervalo 95% (2021-2025)',
    )
)

# Linha Principal Estimativa
fig_prophet.add_trace(
    go.Scatter(
        x=df_est['ds'],
        y=df_est['Previsao_Prophet'],
        mode='lines',
        name='2. Estimativa Prophet (2021-2025)',
        line=dict(color='#FFA500', dash='dot', width=2.5),
    )
)

# -------------------------------------------------------------------------
# FASE 3: PROJEÇÃO PROPHET (2026 - 2027) -> Azul + Faixa 95%
# -------------------------------------------------------------------------
# Faixa de Incerteza (Azul Translúcido)
fig_prophet.add_trace(
    go.Scatter(
        x=pd.concat([df_fut['ds'], df_fut['ds'][::-1]]),
        y=pd.concat([
            df_fut['Prophet_Limite_Superior_95'],
            df_fut['Prophet_Limite_Inferior_95'][::-1],
        ]),
        fill='toself',
        fillcolor='rgba(30, 144, 255, 0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo='skip',
        showlegend=False,
        name='Intervalo 95% (2026-2027)',
    )
)

# Linha Principal Projeção
fig_prophet.add_trace(
    go.Scatter(
        x=df_fut['ds'],
        y=df_fut['Previsao_Prophet'],
        mode='lines',
        name='3. Projeção Prophet (2026-2027)',
        line=dict(color='#1E90FF', dash='dot', width=2.5),  # Azul
    )
)

# -------------------------------------------------------------------------
# ELEMENTOS VISUAIS E DIVISORES DE FASE
# -------------------------------------------------------------------------
fig_prophet.add_vline(
    x='2021-01-01', line_dash='dot', line_color='gray', line_width=1.5
)
fig_prophet.add_vline(
    x='2026-01-01', line_dash='dot', line_color='gray', line_width=1.5
)

fig_prophet.add_annotation(
    x='2021-01-01',
    y=1.02,
    yref='paper',
    text='Início Estimativa',
    showarrow=False,
    xanchor='right',
    font=dict(color='gray', size=11),
)

fig_prophet.add_annotation(
    x='2026-01-01',
    y=1.02,
    yref='paper',
    text='Início Projeção',
    showarrow=False,
    xanchor='right',
    font=dict(color='gray', size=11),
)

# Layout Otimizado
fig_prophet.update_layout(
    title=dict(
        text=(
            '<b>Coleta de RSU em SP: Trajetória Prophet (2013–2027)</b><br><sup>'
            'Histórico Real, Estimativa (2021-2025) e Projeção com Intervalos'
            ' de Confiança (95%)</sup>'
        ),
        font=dict(size=18),
    ),
    xaxis_title='Ano/Mês',
    yaxis_title='Total Geral (Toneladas)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='Gray',
        borderwidth=1,
    ),
    xaxis=dict(
        dtick='M12',
        tickformat='%Y',
        showgrid=True,
        gridcolor='LightGray',
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='LightGray',
        tickformat=',.0f',
    ),
    margin=dict(l=50, r=150, t=90, b=50),
)

fig_prophet.show()

# 9. VISUALIZAÇÃO DEFINITIVA DA PROJEÇÃO PROPHET (2013-2027)

In [21]:
# 1. Isolando o Histórico em Memória (2013-2020)
df_full_train = df_clean[df_clean['ds'] <= '2020-12-01'].sort_values(by='ds')

# 2. Máscaras Temporais sobre o DataFrame Consolidado
est_mask = (df_previsoes_consolidado['ds'] >= '2021-01-01') & (
    df_previsoes_consolidado['ds'] <= '2025-12-01'
)
fut_mask = df_previsoes_consolidado['ds'] >= '2026-01-01'

# 3. Construção do Gráfico
fig = go.Figure()

# FASE 1: Dados Históricos (2013-2020) -> Linha Preta
fig.add_trace(
    go.Scatter(
        x=df_full_train['ds'],
        y=df_full_train['y'],
        mode='lines',
        name='1. Dados Históricos (2013-2020)',
        line=dict(color='black', width=1.5),
    )
)

# FASE 2: Estimativa Prophet (2021-2025) -> Linha Amarela/Laranja
fig.add_trace(
    go.Scatter(
        x=df_previsoes_consolidado.loc[est_mask, 'ds'],
        y=df_previsoes_consolidado.loc[est_mask, 'Previsao_Prophet'],
        mode='lines',
        name='2. Estimativa Prophet (2021-2025)',
        line=dict(color='orange', dash='dot', width=2.5),
    )
)

# FASE 3: Projeção Prophet (2026-2027) -> Linha Azul
fig.add_trace(
    go.Scatter(
        x=df_previsoes_consolidado.loc[fut_mask, 'ds'],
        y=df_previsoes_consolidado.loc[fut_mask, 'Previsao_Prophet'],
        mode='lines',
        name='3. Projeção Prophet (2026-2027)',
        line=dict(color='blue', dash='dot', width=2.5),
    )
)

# Linhas Divisórias de Fase
fig.add_vline(
    x='2021-01-01', line_dash='dot', line_color='gray', line_width=1.5
)
fig.add_vline(
    x='2026-01-01', line_dash='dot', line_color='gray', line_width=1.5
)

# Ajustes de Layout e Formatação
fig.update_layout(
    title=(
        'Coleta de RSU em SP: Dados Históricos, Estimativa e Projeção Prophet'
        ' (2013-2027)'
    ),
    xaxis_title='Tempo (Meses)',
    yaxis_title='Total Geral (Toneladas)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        font=dict(size=12),
        bgcolor='rgba(255, 255, 255, 0.8)',
        bordercolor='Gray',
        borderwidth=1,
    ),
    xaxis=dict(
        title='Tempo (Meses)',
        dtick='M12',
        tickformat='%Y',
        showgrid=True,
        gridcolor='LightGray',
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='LightGray',
        tickformat=',.0f',
    ),
    margin=dict(l=50, r=150, t=80, b=50),
)

fig.show()

# 10. EXPORTAÇÃO DAS PROJEÇÕES CONSOLIDADAS

In [22]:
nome_arquivo_exportacao = "DF/projecoes_consolidadas_2021_2027.xlsx"
df_previsoes_consolidado.to_excel(nome_arquivo_exportacao, index=False)
print(f"Arquivo de projeções salvo com sucesso em: {nome_arquivo_exportacao}")

Arquivo de projeções salvo com sucesso em: DF/projecoes_consolidadas_2021_2027.xlsx
